# HAWQ-v2 vs Uniform Quantization on Tiny-ImageNet

End-to-end pipeline:

1. Load Tiny-ImageNet (200 classes) and DeiT-small (22M params, ImageNet pretrained).
2. Linear-probe the new 200-class head for a couple of epochs so the model actually classifies Tiny-ImageNet.
3. Replace `nn.Linear` blocks with `QLinear` wrappers.
4. Run the **HAWQ-v2 Hutchinson-trace analyzer** to get per-layer sensitivity.
5. Compare three things at matched bit budgets:
   - FP32 baseline
   - **Uniform PTQ** (PyTorch-style: same bit-width everywhere)
   - **HAWQ-v2 mixed precision** (our method: budget-based allocation guided by trace sensitivity)
6. Plot the accuracy / size tradeoff curve.

In [ ]:
import sys
from pathlib import Path
import copy
import json

# Make our local hawq-v2/ files importable as top-level modules.
HERE = Path('.').resolve()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoModelForImageClassification

import data, analyzer_v2, bit_allocator, quantizer, ptq

torch.manual_seed(0)
np.random.seed(0)

device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
print(f'Device: {device}')

## 1. Data

We use a subset of Tiny-ImageNet to keep iteration fast. Bump these numbers up
(or set them to `None`) if running on a beefy GPU.

In [ ]:
TRAIN_SUBSET = 10000   # how many training images to use for the linear probe
VAL_SUBSET   = 2000    # how many validation images for accuracy reporting
BATCH_SIZE   = 64

train_loader, val_loader, num_classes = data.load_tiny_imagenet(
    batch_size=BATCH_SIZE,
    num_workers=2,
    image_size=224,
    train_subset=TRAIN_SUBSET,
    val_subset=VAL_SUBSET,
)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
print(f'Classes: {num_classes}')

## 2. Model

DeiT-small from HuggingFace. We swap its 1000-class head for a fresh
200-class head and quickly linear-probe it on Tiny-ImageNet so the model
actually classifies the right label space.

In [ ]:
model = AutoModelForImageClassification.from_pretrained(
    'facebook/deit-small-patch16-224',
    num_labels=num_classes,
    ignore_mismatched_sizes=True,
    attn_implementation='eager',  # SDPA fused kernel doesn't support 2nd-order autograd (Hutchinson needs it)
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Total params: {n_params:,}')

In [ ]:
# Linear probe: freeze backbone, train only the new classifier head.
for name, p in model.named_parameters():
    p.requires_grad = ('classifier' in name) or ('head' in name)

trainable = [p for p in model.parameters() if p.requires_grad]
print(f'Trainable head params: {sum(p.numel() for p in trainable):,}')

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(trainable, lr=1e-3, weight_decay=1e-4)

EPOCHS = 2
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for inputs, targets in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        optimizer.zero_grad()
        out = model(inputs)
        logits = out.logits if hasattr(out, 'logits') else out
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        running += loss.item()
    print(f'Epoch {epoch+1}/{EPOCHS}: train_loss = {running/len(train_loader):.4f}')

# Now every parameter can have its own gradient again - HAWQ-v2 needs this.
for p in model.parameters():
    p.requires_grad = True

## 3. FP32 baseline accuracy

In [ ]:
fp32_acc = ptq.evaluate(model, val_loader, device=device)
fp32_size_mb = sum(p.numel() for p in model.parameters()) * 4 / (1024 ** 2)
print(f'FP32 accuracy: {fp32_acc*100:.2f}%  |  size: {fp32_size_mb:.2f} MB')

## 4. Wrap target Linear blocks in QLinear

We quantize the transformer's Linear blocks (attention QKV, attention
projection, MLP fc1/fc2). The classification head stays in FP32 so the
comparison is purely about backbone compression.

In [ ]:
target_names = [
    name for name, m in model.named_modules()
    if isinstance(m, nn.Linear)
    and 'classifier' not in name.lower()
    and 'head' not in name.lower()
]
print(f'Wrapping {len(target_names)} linear blocks with QLinear')

quantizer.replace_linear_with_qlinear(model, target_names)
model = model.to(device)

params_per_layer = ptq.collect_qlinear_param_counts(model)
print(f'QLinear blocks now in model: {len(params_per_layer)}')

## 5. HAWQ-v2 sensitivity analysis (Hutchinson trace)

For each QLinear block we estimate `Tr(H)` over a few batches and a few
Rademacher samples per batch. The sensitivity score `S_i = |Tr(H)| / n_i`
is what the allocator consumes.

In [ ]:
from torch.utils.data import DataLoader

# Hutchinson trace needs second-order autograd, which holds the entire forward
# graph in memory. Use a tiny batch just for the analyzer; the rest of the
# pipeline keeps the bigger BATCH_SIZE.
analyzer_loader = DataLoader(
    val_loader.dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

torch.cuda.empty_cache()

analyzer = analyzer_v2.ViTHAWQv2Analyzer(model, analyzer_loader, criterion)

# Pick QLinear modules only (we don't want the head re-included accidentally).
target_blocks = {
    name: m for name, m in model.named_modules()
    if isinstance(m, quantizer.QLinear)
}

sensitivities = analyzer.compute_layer_sensitivities(
    target_blocks=target_blocks,
    num_samples=16,
    num_batches=4,
    sort_desc=True,
    verbose=True,
)

torch.cuda.empty_cache()

In [ ]:
# Dump for the slider notebook.
with open('sensitivities.json', 'w') as f:
    json.dump(sensitivities, f, indent=2)
print(f'Saved sensitivities for {len(sensitivities)} layers -> sensitivities.json')

# Quick sanity plot: distribution of S_i.
names = list(sensitivities.keys())
scores = [sensitivities[n]['S_i'] for n in names]
plt.figure(figsize=(10, 3))
plt.bar(range(len(scores)), scores)
plt.yscale('log')
plt.ylabel('S_i (log)')
plt.xlabel('layer rank (most -> least sensitive)')
plt.title('HAWQ-v2 per-layer sensitivity')
plt.tight_layout()
plt.show()

## 6. The three-way comparison

For each method we record top-1 accuracy and effective model size (MB).
Activation quantization is off so the comparison is purely about weight
compression - same setup that papers report PTQ accuracy at.

In [ ]:
allocator = bit_allocator.HAWQv2BitAllocator(candidate_weight_bits=(8, 6, 4))
results = []

# --- FP32 baseline (no quantization) --------------------------------------
ptq.disable_all_quantization(model)
results.append({
    'method': 'FP32',
    'avg_bits': 32.0,
    'accuracy': fp32_acc,
    'size_mb': fp32_size_mb,
})

# --- Uniform PTQ baselines ------------------------------------------------
# 2-bit dropped from comparison: it's below DeiT-small's breaking point,
# so no allocation strategy can survive having any layer at 2 bits.
for bits in [8, 6, 4]:
    bit_assign = allocator.allocate_uniform(sensitivities, bits)
    ptq.apply_bit_assignment(model, bit_assign, quantize_activations=False)
    acc = ptq.evaluate(model, val_loader, device=device)
    size_mb = ptq.compute_effective_size(model, bit_assign)
    avg_b = ptq.average_bits(bit_assign, params_per_layer)
    results.append({
        'method': f'Uniform-{bits}',
        'avg_bits': avg_b,
        'accuracy': acc,
        'size_mb': size_mb,
    })
    print(f'Uniform-{bits}:  avg={avg_b:.2f}  acc={acc*100:.2f}%  size={size_mb:.2f}MB')

In [ ]:
# --- HAWQ-v2 mixed precision at several budgets ---------------------------
# Budgets stay strictly inside (min, max) of candidate bits so the allocator
# actually has room to mix. With candidates {4, 6, 8} the meaningful range
# is (4, 8); a budget at the extremes degenerates to uniform.
BUDGETS = [4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5]

for target in BUDGETS:
    bit_assign = allocator.allocate_by_budget(sensitivities, target)
    avg_b = ptq.average_bits(bit_assign, params_per_layer)
    ptq.apply_bit_assignment(model, bit_assign, quantize_activations=False)
    acc = ptq.evaluate(model, val_loader, device=device)
    size_mb = ptq.compute_effective_size(model, bit_assign)
    results.append({
        'method': f'HAWQ-v2-{target:g}',
        'avg_bits': avg_b,
        'accuracy': acc,
        'size_mb': size_mb,
    })
    print(f'HAWQ-v2 target={target:g}: actual_avg={avg_b:.2f}  acc={acc*100:.2f}%  size={size_mb:.2f}MB')

In [ ]:
df = pd.DataFrame(results)
df_sorted = df.sort_values('avg_bits').reset_index(drop=True)
print(df_sorted.to_string(index=False))
df_sorted.to_csv('results.csv', index=False)

## 7. Tradeoff plot

Y axis: accuracy. X axis: average bit-width. The whole point of HAWQ-v2
is that the orange curve sits above the blue curve at the same average
bit-width.

In [ ]:
uni = df[df['method'].str.startswith('Uniform')].sort_values('avg_bits')
mix = df[df['method'].str.startswith('HAWQ-v2')].sort_values('avg_bits')

plt.figure(figsize=(8, 5))
plt.plot(uni['avg_bits'], uni['accuracy'] * 100, 'o-', label='Uniform PTQ', linewidth=2)
plt.plot(mix['avg_bits'], mix['accuracy'] * 100, 's-', label='HAWQ-v2 (Hutchinson trace)', linewidth=2)
plt.axhline(fp32_acc * 100, color='gray', linestyle='--', label=f'FP32 ({fp32_acc*100:.1f}%)')
plt.xlabel('Average bit-width')
plt.ylabel('Top-1 Accuracy (%)')
plt.title('HAWQ-v2 vs Uniform PTQ on Tiny-ImageNet (DeiT-small)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('tradeoff.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Size-vs-accuracy view (the angle the paper cares about for memory budgets).
plt.figure(figsize=(8, 5))
plt.plot(uni['size_mb'], uni['accuracy'] * 100, 'o-', label='Uniform PTQ', linewidth=2)
plt.plot(mix['size_mb'], mix['accuracy'] * 100, 's-', label='HAWQ-v2 (Hutchinson trace)', linewidth=2)
plt.scatter([fp32_size_mb], [fp32_acc * 100], color='gray', marker='*', s=150, label=f'FP32 ({fp32_size_mb:.1f}MB)')
plt.xlabel('Effective model size (MB)')
plt.ylabel('Top-1 Accuracy (%)')
plt.title('Accuracy vs Model Size on Tiny-ImageNet')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('size_vs_acc.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Save model state for the slider notebook

In [ ]:
ptq.disable_all_quantization(model)
torch.save({
    'state_dict': model.state_dict(),
    'target_names': target_names,
    'sensitivities': sensitivities,
    'fp32_acc': fp32_acc,
    'fp32_size_mb': fp32_size_mb,
}, 'checkpoint.pt')
print('Saved checkpoint.pt')